In [1]:
from dotenv import load_dotenv
import os
from openai import OpenAI
import json
import gradio as gr
from pydantic import BaseModel
from pypdf import PdfReader
import requests

In [2]:
load_dotenv(override=True)
openai = OpenAI()

In [3]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if not pushover_user or not pushover_token:
    raise ValueError("Pushover credentials are not set in environment variables.")
else:
    print("Pushover credentials loaded successfully.")
    

Pushover credentials loaded successfully.


In [4]:
def push(message):
    print(f"Push: {message}")
    payload = {
        "user": pushover_user,
        "token": pushover_token,
        "message": message
    }
    response = requests.post(pushover_url, data=payload)
    if response.status_code == 200:
        print("Notification sent successfully.")
    else:
        print("Failed to send notification.")

In [5]:
push("Hello from the AI Engineer Agent!")

Push: Hello from the AI Engineer Agent!
Notification sent successfully.


In [6]:
def record_user_details(email, name="Name not provided", notes="No additional notes"):
    if not name:
        name = "Name not provided"
    if not notes:
        notes = "No additional notes"

    push(f"New user details recorded:\nEmail: {email}\nName: {name}\nNotes: {notes}")

    return {
        "status": "success",
        "message": "User details recorded and notification sent."
    }

def record_unknown_question(question):
    push(f"Unknown question received: {question} that I couldn't answer. Please review and update the knowledge base.")
    return {"status": "success", "message": "Unknown question recorded and notification sent."}

In [7]:
record_user_details_json = {
    "name": "record_user_details", 
    "description": "Use this tool when a user provides contact information or wants to be contacted. Extract and include the user's email, name if mentioned, and useful notes about why they want to connect. If the user provides a name or reason, do not omit them.",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "description": "The email address of the user."
            },
            "name": {
                "type": "string",
                "description": "The user's name if mentioned in the conversation. If not mentioned, use 'Name not provided'."
            },
            "notes": {
                "type": "string",
                "description": "Useful context about why the user wants to connect, what they are interested in, or what they asked about. If no context is available, use 'No additional notes'."
            }
        },
        "required": ["email", "name", "notes"],
        "additionalProperties": False
    }
}


record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "use this tool to record that a question was asked that the agent didn't know how to answer. This will help us identify gaps in the knowledge base and improve the agent's performance over time.",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The question that the agent was unable to answer."
            }
        },
        "required": ["question"],
        "additionalProperties": False
    }
}


tools = [
    {"type": "function", "function": record_user_details_json},
    {"type": "function", "function": record_unknown_question_json}
]

In [8]:
tools


[{'type': 'function',
  'function': {'name': 'record_user_details',
   'description': "Use this tool when a user provides contact information or wants to be contacted. Extract and include the user's email, name if mentioned, and useful notes about why they want to connect. If the user provides a name or reason, do not omit them.",
   'parameters': {'type': 'object',
    'properties': {'email': {'type': 'string',
      'description': 'The email address of the user.'},
     'name': {'type': 'string',
      'description': "The user's name if mentioned in the conversation. If not mentioned, use 'Name not provided'."},
     'notes': {'type': 'string',
      'description': "Useful context about why the user wants to connect, what they are interested in, or what they asked about. If no context is available, use 'No additional notes'."}},
    'required': ['email', 'name', 'notes'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'record_unknown_question',
   

In [9]:
# This function can take a list of tool calls, and run them. This is the IF statement!!

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Handling tool call: {tool_name} with arguments: {arguments}", flush=True)
        
        if tool_name == "record_user_details":
            result = record_user_details(**arguments)
        elif tool_name == "record_unknown_question":
            result = record_unknown_question(**arguments)

        results.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tool_call.id})
    return results

In [10]:
globals()["record_unknown_question"]("this is really hard question that I don't know how to answer")

Push: Unknown question received: this is really hard question that I don't know how to answer that I couldn't answer. Please review and update the knowledge base.
Notification sent successfully.


{'status': 'success',
 'message': 'Unknown question recorded and notification sent.'}

In [11]:
# This is a more elegant way that avoids the IF statement.

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Handling tool call: {tool_name} with arguments: {arguments}", flush=True)
        
        # Dynamically call the function based on its name
        if tool_name in globals():
            tool = globals().get(tool_name)
            result = tool(**arguments) if tool else {"status": "error", "message": f"Tool {tool_name} not found."}
            results.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tool_call.id})
    return results

In [12]:
reader = PdfReader("me/sahil_linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text + "\n"
        
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()
    
name = "Sahil" 

In [13]:
system_prompt = f"""You are acting as {name}. You are answering questions on {name}'s website,
particularly questions related to {name}'s career, background, skills and experience.

Your responsibility is to represent {name} for interactions on the website as faithfully as possible.

You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions.

Be professional and engaging, as if talking to a potential client or future employer who came across the website.

If you don't know the answer to any question, use your record_unknown_question tool to record the question.

If the user provides contact details or wants to get in touch, use the record_user_details tool.
When calling record_user_details, always extract:
1. email address
2. user's name, if mentioned
3. notes explaining why they want to connect or what they are interested in

Do not omit name or notes if the user provided them.
If name is missing, use "Name not provided".
If notes are missing, use "No additional notes".
"""

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [14]:

def chat(message, history):
    if history is None:
        history = []

    messages = [{"role": "system", "content": system_prompt}]

    # Convert old history into OpenAI message format
    for user_msg, assistant_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": assistant_msg})

    messages.append({"role": "user", "content": message})

    done = False

    while not done:
        response = openai.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools
        )

        finish_reason = response.choices[0].finish_reason

        if finish_reason == "tool_calls":
            assistant_message = response.choices[0].message
            tool_calls = assistant_message.tool_calls

            tool_results = handle_tool_calls(tool_calls)

            messages.append(assistant_message)
            messages.extend(tool_results)

        else:
            done = True

    reply = response.choices[0].message.content

    history.append((message, reply))

    return reply, history

In [15]:
gr.Interface(
    fn=chat,
    inputs=["text", "state"],
    outputs=["text", "state"]
).launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


In [ ]:
# I liked your profile. My email is john@gmail.com.

# I liked your profile. My email is john@gmail.com.  What is Sahil's GPA in Master's in the 3rd semester?

Handling tool call: record_user_details with arguments: {'email': 'john@gmail.com', 'name': 'Name not provided', 'notes': "inquired about Sahil's GPA in Master's 3rd semester."}
Push: New user details recorded:
Email: john@gmail.com
Name: Name not provided
Notes: inquired about Sahil's GPA in Master's 3rd semester.
Notification sent successfully.
Handling tool call: record_unknown_question with arguments: {'question': "What is Sahil's GPA in Master's in the 3rd semester?"}
Push: Unknown question received: What is Sahil's GPA in Master's in the 3rd semester? that I couldn't answer. Please review and update the knowledge base.
Notification sent successfully.
Using existing dataset file at: .gradio\flagged\dataset1.csv
